# Hippo acoustic warning: quickstart

Two physical facts behind the repository, computed in under a minute with no data download:

1. how little of an underwater call crosses the water surface into the air;
2. why a shallow pool or river does not carry the low-frequency part of the call far.

Everything here calls the public code in `src/`. Run the cells top to bottom.

In [ ]:
# Setup: works in Google Colab and in a local clone of the repository.
import os, sys
if not os.path.isdir("src"):
    !git clone -q https://github.com/drdmitrymikhaylov/acousticpinn-hippo-attack-warning.git
    %cd acousticpinn-hippo-attack-warning
!pip -q install numpy scipy matplotlib
sys.path.insert(0, "src")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import propagation as P
import waveguide_cutoff as W

print(f"Water -> air interface loss: {P.interface_loss_db():.1f} dB")
print(f"Refraction cone in air: {P.refraction_cone_deg():.1f} degrees half-angle")

## 1. The water surface is almost a mirror

Power transmission from water into air against the angle of incidence. Beyond the critical angle nothing gets through.

In [ ]:
theta = np.linspace(0, np.deg2rad(20), 400)
T = np.array([P.transmission_coefficient(t) for t in theta])
plt.figure(figsize=(6, 3.5))
plt.semilogy(np.rad2deg(theta), np.maximum(T, 1e-12))
plt.xlabel("incidence angle in water (degrees)")
plt.ylabel("power transmitted into air")
plt.title("Water to air, plane wave")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Shallow water has a lowest frequency it will carry

Cut-off of the first mode over a rigid floor, and the extra loss below cut-off compared with the simple spreading law used for range budgets.

In [ ]:
for H in (1, 2, 3, 5):
    print(f"depth {H} m: first-mode cut-off {W.cutoff_rigid_hz(H):6.0f} Hz (rigid floor), "
          f"{W.cutoff_fluid_bottom_hz(H, 1700.0):6.0f} Hz (saturated sand)")

H, zs, zr = 3.0, 1.5, 1.5
r = np.linspace(3, 30, 60)
plt.figure(figsize=(6, 3.5))
for f in (50.0, 205.0):
    exact = [W.tl_exact_db(f, H, zs, zr, ri) for ri in r]
    plt.plot(r, exact, label=f"{f:.0f} Hz, exact field")
plt.plot(r, [P.tl_water_db(ri, depth_m=H) for ri in r], "k--", label="spreading law")
plt.gca().invert_yaxis()
plt.xlabel("range (m)"); plt.ylabel("transmission loss re 1 m (dB)")
plt.title("3 m of water, source and hydrophone at mid-depth")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

Below cut-off (50 Hz here) the field decays exponentially and falls far below the range budget; above it (205 Hz) the guide is close to the law. The full discussion is in the README section *A shallow pool does not carry the low band*.